In [1]:
import os

import pandas as pd
import matplotlib.pyplot as plt
from pyvis.network import Network
from bokeh.plotting import figure, from_networkx, output_file, save
from bokeh.models import Circle, MultiLine
from bokeh.io import output_file, save

import psycopg2
import networkx as nx

In [10]:
ada = pd.read_csv("../data/rtc/ada_cp2021.csv", sep=";")
attivita = pd.read_csv("../data/rtc/attivita_ada.csv", sep=";")

In [4]:
def build_graph_from_dataframe(df, column1, column2):
    """Build bipartite graph from ADA and attivita dataframes."""
    G = nx.Graph()
    
    # Filter out nulls
    edges_df = df[[column1, column2]].dropna()
    

    # Add edges
    G.add_edges_from([
        (f"{row[column2]}", f"{row[column1]}") 
        for _, row in edges_df.iterrows()
    ])
    
    # Get column2 IDs from edges
    column2_ids = edges_df[column2].unique()
    
    # Add node attributes (blue)
    for id2 in column2_ids:
        node_id = f"{id2}"
        if node_id in G.nodes:
            # Get label
            row = df[df[column2] == id2]
            if not row.empty:
                label = row.iloc[0]['titolo_ada']
            else:
                label = f"{id2}"
            
            G.nodes[node_id]['type'] = 'ADA' # change to type of datapoint
            G.nodes[node_id]['color'] = 'blue'
            G.nodes[node_id]['label'] = label
            G.nodes[node_id]['title'] = f"ADA: {label}\nID: {id2}"
    
    # Get column1 IDs from edges
    column1_ids = edges_df[column1].unique()
    
    # Add ADA node attributes (red)
    for id1 in column1_ids:
        node_id = f"{id1}"
        if node_id in G.nodes:
            G.nodes[node_id]['type'] = 'CP2021' # change to type of datapoint
            G.nodes[node_id]['color'] = 'red'
            G.nodes[node_id]['label'] = f"{id1}"
            G.nodes[node_id]['title'] = f"CP2021: {id1}"
    
    return G

In [ ]:
# # Method 2: Pandas approach (easier for complex queries)
# def build_graph_pandas(conn):
#     query = """
#     SELECT id_profilo as source, id_competenza as target
#     FROM competenze 
#     WHERE id_profilo IS NOT NULL AND id_competenza IS NOT NULL
#     """
    
#     df = pd.read_sql_query(query, conn)
#     G = nx.from_pandas_edgelist(df, source='source', target='target')
    
#     # Add node types
#     profili_df = pd.read_sql_query("SELECT DISTINCT id_profilo as id FROM competenze WHERE id_profilo IS NOT NULL", conn)
#     for id in profili_df['id']:
#         G.nodes[id]['type'] = 'profilo'
        
#     comp_df = pd.read_sql_query("SELECT DISTINCT id_competenza as id FROM competenze WHERE id_competenza IS NOT NULL", conn)
#     for id in comp_df['id']:
#         G.nodes[id]['type'] = 'competenza'
    
#     return G

In [ ]:
df = ada
column1 = 'codice_ada'
column2 = 'codice_cp2021'

# Build graph from dataframes
G = build_graph_from_dataframe(ada, column1, column2)

Nodes: 1585
Edges: 1859
Attivita nodes: 0
ADA nodes: 629
Sample edges: [('2.3.1.3.0', 'ADA.01.01.01'), ('2.3.1.3.0', 'ADA.01.01.02'), ('2.3.1.3.0', 'ADA.01.01.09'), ('2.3.1.3.0', 'ADA.01.01.15'), ('2.3.1.3.0', 'ADA.01.01.16')]


In [12]:
# Add more edges from attivita dataframe
additional_edges = attivita[['id_attivita', 'codice_ada']].dropna()
for _, row in additional_edges.iterrows():
    attivita_node = f"A_{row['id_attivita']}"
    ada_node = f"ADA_{row['codice_ada']}"
    
    # Add edge
    G.add_edge(attivita_node, ada_node)
    
    # Add attivita node attributes if new
    if attivita_node not in G.nodes or 'color' not in G.nodes[attivita_node]:
        G.nodes[attivita_node]['type'] = 'attivita'
        G.nodes[attivita_node]['color'] = 'green'
        G.nodes[attivita_node]['label'] = row.get('attivita', f"Attivita {row['id_attivita']}")
        G.nodes[attivita_node]['title'] = f"Attivita: {row.get('attivita', row['id_attivita'])}\nID: {row['id_attivita']}"
    
    # Add ADA node attributes if new
    if ada_node not in G.nodes or 'color' not in G.nodes[ada_node]:
        G.nodes[ada_node]['type'] = 'ada'
        G.nodes[ada_node]['color'] = 'blue'
        G.nodes[ada_node]['label'] = f"ADA {row['codice_ada']}"
        G.nodes[ada_node]['title'] = f"Codice ADA: {row['codice_ada']}"
        
# Inspect the graph
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
print(f"Attivita nodes: {sum(1 for n in G.nodes() if G.nodes[n].get('type') == 'attivita')}")
print(f"ADA nodes: {sum(1 for n in G.nodes() if G.nodes[n].get('type') == 'ADA')}")
print(f"CP2021 nodes: {sum(1 for n in G.nodes() if G.nodes[n].get('type') == 'CP2021')}")
print("Sample edges:", list(G.edges())[:5])

Nodes: 9865
Edges: 9178
Attivita nodes: 7319
ADA nodes: 629
CP2021 nodes: 956
Sample edges: [('2.3.1.3.0', 'ADA.01.01.01'), ('2.3.1.3.0', 'ADA.01.01.02'), ('2.3.1.3.0', 'ADA.01.01.09'), ('2.3.1.3.0', 'ADA.01.01.15'), ('2.3.1.3.0', 'ADA.01.01.16')]


In [13]:
# Create graph with pyvis
net = Network(height='750px', width='100%', notebook=True)
net.from_nx(G)
net.save_graph(f'/app/ada_attivita_cp2021_graph.html')
print(f"Graph saved to /app/ada_attivita_cp2021_graph.html")

Graph saved to /app/ada_attivita_cp2021_graph.html
